In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install -U duckdb

In [ ]:
import zipfile
from pathlib import Path

zip_path = "/content/drive/MyDrive/ProjetoDados/arquivoDB.zip"
destino = "/content/arquivoDB_extraido"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(destino)

for p in Path(destino).rglob("*"):
    if p.is_file():
        print(p)

/content/arquivoDB_extraido/discord_projeto.duckdb


In [ ]:
import duckdb

db_path = "/content/arquivoDB_extraido/discord_projeto.duckdb"
csv_path = "/content/guild_ids.csv"

con = duckdb.connect(db_path, read_only=True)

# Identifica automaticamente a tabela que contém guild_id
tabelas = con.execute("SHOW TABLES").fetchall()
tabela_escolhida = None

for (tabela,) in tabelas:
    colunas = con.execute(
        f'DESCRIBE "{tabela}"'
    ).fetchdf()["column_name"].tolist()

    if "guild_id" in colunas:
        tabela_escolhida = tabela
        break

if tabela_escolhida is None:
    raise ValueError("Nenhuma tabela com a coluna guild_id foi encontrada.")

# Exporta os IDs únicos, excluindo valores nulos
con.execute(f"""
    COPY (
        SELECT DISTINCT guild_id
        FROM "{tabela_escolhida}"
        WHERE guild_id IS NOT NULL
        ORDER BY guild_id
    )
    TO '{csv_path}'
    (HEADER, DELIMITER ',')
""")

total = con.execute(f"""
    SELECT COUNT(*)
    FROM read_csv_auto('{csv_path}')
""").fetchone()[0]

con.close()

print(f"Tabela utilizada: {tabela_escolhida}")
print(f"Servidores únicos exportados: {total}")
print(f"CSV salvo em: {csv_path}")

Tabela utilizada: dataset
Servidores únicos exportados: 100
CSV salvo em: /content/guild_ids.csv


In [ ]:
from google.colab import files
files.download("/content/guild_ids.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>